# Pharmacological Dose–Response Analysis

**Portfolio demonstration — simulated data, not experimental or clinical evidence.**

## Research question
Can a four-parameter logistic (4PL) model describe a simulated concentration–response relationship, and how can the half-maximal effective concentration (EC50) be estimated?

## Methods
- Generate a transparent synthetic concentration–response dataset.
- Fit a 4PL model with `scipy.optimize.curve_fit`.
- Estimate EC50 and inspect residuals.
- Plot observed values and the fitted curve.

The simulated values are designed for learning. They must not be interpreted as measurements of a real drug.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import pearsonr

rng = np.random.default_rng(42)
concentration = np.logspace(-2, 2, 16)  # arbitrary concentration units
bottom, top, log_ec50_true, hill_true = 4.0, 96.0, 0.65, 1.15

def four_parameter_logistic(x, bottom, top, log_ec50, hill):
    log_x = np.log10(x)
    return bottom + (top - bottom) / (1 + 10 ** ((log_ec50 - log_x) * hill))

response_true = four_parameter_logistic(
    concentration, bottom, top, log_ec50_true, hill_true
)
response_observed = np.clip(response_true + rng.normal(0, 3, len(concentration)), 0, 100)

data = pd.DataFrame({
    "concentration_arb_units": concentration,
    "response_percent": response_observed
})
data.head()

In [ ]:
initial_guess = [0, 100, 0, 1]
bounds = ([-20, 50, -4, 0.1], [50, 130, 4, 4])

parameters, covariance = curve_fit(
    four_parameter_logistic,
    concentration,
    response_observed,
    p0=initial_guess,
    bounds=bounds,
    maxfev=20000
)
bottom_fit, top_fit, log_ec50_fit, hill_fit = parameters
ec50_fit = 10 ** log_ec50_fit

predicted = four_parameter_logistic(concentration, *parameters)
residuals = response_observed - predicted
rmse = float(np.sqrt(np.mean(residuals ** 2)))

print(f"Estimated EC50: {ec50_fit:.4g} arbitrary concentration units")
print(f"Estimated Hill slope: {hill_fit:.3f}")
print(f"RMSE: {rmse:.3f} response percentage points")

In [ ]:
x_smooth = np.logspace(np.log10(concentration.min()), np.log10(concentration.max()), 300)
plt.figure(figsize=(8, 5))
plt.scatter(concentration, response_observed, label="Simulated observations")
plt.plot(x_smooth, four_parameter_logistic(x_smooth, *parameters),
         label="Fitted 4PL model")
plt.xscale("log")
plt.xlabel("Concentration (arbitrary units; log scale)")
plt.ylabel("Response (%)")
plt.title("Simulated concentration–response analysis")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Interpretation and limitations

The fitted EC50 is an estimate for this simulated dataset only. EC50 depends on the assay system, endpoint, experimental conditions, and model assumptions; it is not automatically equivalent to binding affinity or clinical dose.

Important limitations:
- The data are synthetic and contain no biological replicates.
- The 4PL model assumes a sigmoidal response and may be inappropriate for other response shapes.
- Parameter uncertainty and model diagnostics should be examined before drawing scientific conclusions.
- This notebook is an educational coding demonstration, not pharmacological evidence or medical advice.

## Reproducibility
The random number generator uses a fixed seed (`42`). Run the notebook from top to bottom in a Python 3 environment with NumPy, pandas, SciPy, and Matplotlib installed.